# Build sum-of-products KEO for formaldehyde ($\mathrm{H_2CO}$) in valence coordinates

In [1]:
import jax
import numpy as np
from jax import numpy as jnp
from scipy import constants

from keosop.gmat import Gmat
from keosop.keo_jax import batch_Gmat, com
from keosop.revad import dihedral, dot, norm

jax.config.update("jax_enable_x64", True)

G_to_invcm = (
    constants.value("Planck constant")
    * constants.value("Avogadro constant")
    * 1e16
    / (4.0 * np.pi**2 * constants.value("speed of light in vacuum"))
    * 1e5
)

Define the internal-to-Cartesian and Cartesian-to-internal coordinate transformations for $\mathrm{H_2CO}$.

The internal coordinates are defined as:

1. $r_{\mathrm{CO}}$ — C–O bond length,
2. $r_{\mathrm{CH_1}}$ — C–H1 bond length,
3. $r_{\mathrm{CH_2}}$ — C–H2 bond length,
4. $\alpha_{\mathrm{H_1CO}}$ — H1–C–O bond angle,
5. $\alpha_{\mathrm{H_2CO}}$ — H2–C–O bond angle,
6. $\Delta\tau = \tau + \pi$, where $\tau$ is the book angle (dihedral angle) between the (H1CO) and (H2CO) planes.

The atom ordering in Cartesian coordinates is:

$$
\mathrm{C},\ \mathrm{O},\ \mathrm{H_1},\ \mathrm{H_2}
$$

In [ ]:
masses = np.array([12.0, 15.99491463, 1.00782505, 1.00782505])  # masses of C, O, H, H

ncoo = 6


def internal_to_cartesian(q):
    rCO, rCH1, rCH2, aOCH1, aOCH2, dtau = q
    tau = np.pi + dtau
    xyz = [
        [0.0, 0.0, 0.0],
        [0.0, 0.0, rCO],
        [
            rCH1 * np.sin(aOCH1) * np.cos(tau * 0.5),
            -rCH1 * np.sin(aOCH1) * np.sin(tau * 0.5),
            rCH1 * np.cos(aOCH1),
        ],
        [
            rCH2 * np.sin(aOCH2) * np.cos(tau * 0.5),
            rCH2 * np.sin(aOCH2) * np.sin(tau * 0.5),
            rCH2 * np.cos(aOCH2),
        ],
    ]
    return np.array(xyz, dtype=object)


def cartesian_to_internal(xyz):
    C, O, H1, H2 = xyz

    CO = O - C
    CH1 = H1 - C
    CH2 = H2 - C

    rCO = norm(CO)
    rCH1 = norm(CH1)
    rCH2 = norm(CH2)

    aOCH1 = np.arccos(dot(CO, CH1) / (rCO * rCH1))
    aOCH2 = np.arccos(dot(CO, CH2) / (rCO * rCH2))

    tau = dihedral(H1, C, O, H2)
    dtau = tau + np.pi

    return np.array([rCO, rCH1, rCH2, aOCH1, aOCH2, dtau], dtype=object)

Define one-dimensional grids for the internal coordinate representation and construct the G-matrix in sum-of-products form

In [3]:
grids = [
    np.linspace(0.8, 1.2, 5),  # r_{CO}
    np.linspace(0.8, 1.2, 5),  # r_{CH1}
    np.linspace(0.8, 1.2, 5),  # r_{CH2}
    np.linspace(60 * np.pi / 180, 160 * np.pi / 180, 5),  # alpha_{H1CO}
    np.linspace(60 * np.pi / 180, 160 * np.pi / 180, 5),  # alpha_{H2CO}
    np.linspace(-60 * np.pi / 180, 60 * np.pi / 180, 5),  # dtau
]

gmat = Gmat(grids, masses, internal_to_cartesian, cartesian_to_internal)
gvib_sop, grovib_sop, grot_sop = gmat.run(return_rot=True)

print("Number of terms in Gvib_{i,j}\n i  j no.terms")
for i in range(ncoo):
    for j in range(i, ncoo):
        print(f"{i:2d} {j:2d} {gvib_sop[i, j].nterms:6d}")

print("\nNumber of terms in Grovib_{i,j}\n i  j no.terms")
for i in range(3):
    for j in range(ncoo):
        print(f"{i:2d} {j:2d} {grovib_sop[i, j].nterms:6d}")

print("\nNumber of terms in Grot_{i,j}\n i  j no.terms")
for i in range(3):
    for j in range(i, 3):
        print(f"{i:2d} {j:2d} {grot_sop[i, j].nterms:6d}")

Number of terms in Gvib_{i,j}
 i  j no.terms
 0  0      1
 0  1      1
 0  2      1
 0  3      2
 0  4      2
 0  5      1
 1  1      1
 1  2      3
 1  3      4
 1  4      6
 1  5      2
 2  2      1
 2  3      6
 2  4      4
 2  5      2
 3  3      7
 3  4     12
 3  5      4
 4  4      7
 4  5      4
 5  5     14

Number of terms in Grovib_{i,j}
 i  j no.terms
 0  0      1
 0  1      1
 0  2      1
 0  3      2
 0  4      2
 0  5      4
 1  0      1
 1  1      1
 1  2      1
 1  3      2
 1  4      2
 1  5      4
 2  0      1
 2  1      2
 2  2      2
 2  3      6
 2  4      6
 2  5      7

Number of terms in Grot_{i,j}
 i  j no.terms
 0  0      1
 0  1      1
 0  2      4
 1  1      1
 1  2      4
 2  2     14


Evaluate the G-matrix numerically on the multidimensional grid formed by the Cartesian product of the one-dimensional grids used to construct the sum-of-products representation

In [4]:
@com(masses)
def internal_to_cartesian_jax(q):
    rCO, rCH1, rCH2, aOCH1, aOCH2, dtau = q
    tau = jnp.pi + dtau
    xyz = [
        [0.0, 0.0, 0.0],
        [0.0, 0.0, rCO],
        [
            rCH1 * jnp.sin(aOCH1) * jnp.cos(tau * 0.5),
            -rCH1 * jnp.sin(aOCH1) * jnp.sin(tau * 0.5),
            rCH1 * jnp.cos(aOCH1),
        ],
        [
            rCH2 * jnp.sin(aOCH2) * jnp.cos(tau * 0.5),
            rCH2 * jnp.sin(aOCH2) * jnp.sin(tau * 0.5),
            rCH2 * jnp.cos(aOCH2),
        ],
    ]
    return jnp.array(xyz, dtype=jnp.float64)


grid_ndim = np.stack(np.meshgrid(*grids, indexing="ij"), axis=-1)
grid_ndim = grid_ndim.reshape(-1, len(grids))

gmat = batch_Gmat(grid_ndim, masses, internal_to_cartesian_jax)

gvib_ndim = gmat[:, :ncoo, :ncoo]
grovib_ndim = gmat[:, ncoo : ncoo + 3, :ncoo]
grot_ndim = gmat[:, ncoo : ncoo + 3, ncoo : ncoo + 3]

Calculate point-wise differences between the sum-of-products and numerical G-matrix

In [5]:
# number of multidimensional grid points
ngrid = grid_ndim.shape[0]

# evaluate the sum-of-products G-matrix on the full direct-product grid
gvib_sop_eval = np.zeros((ngrid, ncoo, ncoo), dtype=np.float64)
grot_sop_eval = np.zeros((ngrid, 3, 3), dtype=np.float64)
grovib_sop_eval = np.zeros((ngrid, 3, ncoo), dtype=np.float64)

# map each multidimensional grid point to its 1D grid indices
grid_indices = np.indices([len(grid) for grid in grids]).reshape(len(grids), -1).T

# Gvib
for ipoint, inds in enumerate(grid_indices):
    for i in range(ncoo):
        for j in range(ncoo):
            sop = gvib_sop[i, j]

            value = 0.0

            for iterm in range(sop.nterms):
                term = 1.0

                for k in range(ncoo):
                    term *= sop.terms[iterm][k][inds[k]]

                value += term

            gvib_sop_eval[ipoint, i, j] = value * G_to_invcm

# Grot
for ipoint, inds in enumerate(grid_indices):
    for i in range(3):
        for j in range(3):
            sop = grot_sop[i, j]

            value = 0.0

            for iterm in range(sop.nterms):
                term = 1.0

                for k in range(ncoo):
                    term *= sop.terms[iterm][k][inds[k]]

                value += term

            grot_sop_eval[ipoint, i, j] = value * G_to_invcm

# Grovib
for ipoint, inds in enumerate(grid_indices):
    for i in range(3):
        for j in range(ncoo):
            sop = grovib_sop[i, j]

            value = 0.0

            for iterm in range(sop.nterms):
                term = 1.0

                for k in range(ncoo):
                    term *= sop.terms[iterm][k][inds[k]]

                value += term

            grovib_sop_eval[ipoint, i, j] = value * G_to_invcm

In [6]:
# differences

diff_vib = gvib_sop_eval - gvib_ndim
diff_rovib = grovib_sop_eval - grovib_ndim
diff_rot = grot_sop_eval - grot_ndim

abs_diff_vib = np.abs(diff_vib)
abs_diff_rovib = np.abs(diff_rovib)
abs_diff_rot = np.abs(diff_rot)

print("--- Gvib ---")
print("Maximum absolute difference:", np.max(abs_diff_vib))
print("Mean absolute difference:", np.mean(abs_diff_vib))
print("RMS difference:", np.sqrt(np.mean(diff_vib**2)))

print("\n--- Grovib ---")
print("Maximum absolute difference:", np.max(abs_diff_rovib))
print("Mean absolute difference:", np.mean(abs_diff_rovib))
print("RMS difference:", np.sqrt(np.mean(diff_rovib**2)))

print("\n--- Grot ---")
print("Maximum absolute difference:", np.max(abs_diff_rot))
print("Mean absolute difference:", np.mean(abs_diff_rot))
print("RMS difference:", np.sqrt(np.mean(diff_rot**2)))

--- Gvib ---
Maximum absolute difference: 3.296918293926865e-12
Mean absolute difference: 6.818715827401518e-15
RMS difference: 4.1351575754844234e-14

--- Grovib ---
Maximum absolute difference: 1.0045078494872343e-10
Mean absolute difference: 1.39659512156936e-12
RMS difference: 6.056230507322969e-12

--- Grot ---
Maximum absolute difference: 5.644551492878236e-11
Mean absolute difference: 1.872460571834662e-12
RMS difference: 5.3154779207332945e-12


In [7]:
print("--- Gvib element-wise errors ---")

for i in range(ncoo):
    for j in range(i, ncoo):
        d = diff_vib[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))

        print(
            f"Gvib[{i},{j}]  "
            f"max abs = {max_abs:.3e}   "
            f"RMS = {rms:.3e}   "
        )

print("\n--- Grovib element-wise errors ---")

for i in range(3):
    for j in range(ncoo):
        d = diff_rovib[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))

        print(
            f"Grovib[{i},{j}]  "
            f"max abs = {max_abs:.3e}   "
            f"RMS = {rms:.3e}   "
        )

print("\n--- Grot element-wise errors ---")

for i in range(3):
    for j in range(i, 3):
        d = diff_rot[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))

        print(
            f"Grot[{i},{j}]  "
            f"max abs = {max_abs:.3e}   "
            f"RMS = {rms:.3e}   "
        )

--- Gvib element-wise errors ---
Gvib[0,0]  max abs = 2.665e-15   RMS = 1.713e-15   
Gvib[0,1]  max abs = 1.554e-15   RMS = 4.961e-16   
Gvib[0,2]  max abs = 1.332e-15   RMS = 5.454e-16   
Gvib[0,3]  max abs = 3.997e-15   RMS = 1.086e-15   
Gvib[0,4]  max abs = 5.329e-15   RMS = 1.082e-15   
Gvib[0,5]  max abs = 1.049e-14   RMS = 8.998e-16   
Gvib[1,1]  max abs = 3.553e-14   RMS = 1.155e-14   
Gvib[1,2]  max abs = 2.665e-15   RMS = 6.490e-16   
Gvib[1,3]  max abs = 2.243e-14   RMS = 4.692e-15   
Gvib[1,4]  max abs = 7.994e-15   RMS = 1.407e-15   
Gvib[1,5]  max abs = 1.119e-13   RMS = 5.699e-15   
Gvib[2,2]  max abs = 2.842e-14   RMS = 1.228e-14   
Gvib[2,3]  max abs = 7.994e-15   RMS = 1.524e-15   
Gvib[2,4]  max abs = 2.220e-14   RMS = 4.667e-15   
Gvib[2,5]  max abs = 7.816e-14   RMS = 5.478e-15   
Gvib[3,3]  max abs = 9.948e-14   RMS = 2.102e-14   
Gvib[3,4]  max abs = 4.796e-14   RMS = 6.711e-15   
Gvib[3,5]  max abs = 2.416e-13   RMS = 1.650e-14   
Gvib[4,4]  max abs = 9.948e-14 

In [8]:
print("--- Gvib largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_vib), abs_diff_vib.shape)
ipoint, i, j = idx

print("grid point :", ipoint)
print("coordinates:", np.asarray(grid_ndim[ipoint]))
print("element    :", (i, j))
print("SOP        :", gvib_sop_eval[ipoint, i, j])
print("reference  :", gvib_ndim[ipoint, i, j])
print("difference :", diff_vib[ipoint, i, j])

print("\n--- Grovib largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_rovib), abs_diff_rovib.shape)
ipoint, i, j = idx

print("grid point :", ipoint)
print("coordinates:", np.asarray(grid_ndim[ipoint]))
print("element    :", (i, j))
print("SOP        :", grovib_sop_eval[ipoint, i, j])
print("reference  :", grovib_ndim[ipoint, i, j])
print("difference :", diff_rovib[ipoint, i, j])

print("\n--- Grot largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_rot), abs_diff_rot.shape)
ipoint, i, j = idx

print("grid point :", ipoint)
print("coordinates:", np.asarray(grid_ndim[ipoint]))
print("element    :", (i, j))
print("SOP        :", grot_sop_eval[ipoint, i, j])
print("reference  :", grot_ndim[ipoint, i, j])
print("difference :", diff_rot[ipoint, i, j])

--- Gvib largest discrepancy ---
grid point : 8170
coordinates: [ 1.          1.1         0.8         1.48352986  2.7925268  -1.04719755]
element    : (np.int64(5), np.int64(5))
SOP        : 622.0418790051233
reference  : 622.04187900512
difference : 3.296918293926865e-12

--- Grovib largest discrepancy ---
grid point : 120
coordinates: [ 0.8         0.8         0.8         2.7925268   2.7925268  -1.04719755]
element    : (np.int64(2), np.int64(5))
SOP        : 1.0043428768429631e-10
reference  : -1.6497264427113395e-14
difference : 1.0045078494872343e-10

--- Grot largest discrepancy ---
grid point : 609
coordinates: [0.8        0.8        1.2        2.7925268  1.48352986 1.04719755]
element    : (np.int64(2), np.int64(2))
SOP        : 156.03359817991188
reference  : 156.03359817996832
difference : -5.644551492878236e-11
